# Semana 3: Modelos Supervisados Clásicos (k-NN y Regresión Logística)
## Caso Práctico Guiado: Detección de Phishing

---

**Objetivo Académico:** En esta sesión, aplicaremos el algoritmo de *k-Nearest Neighbors* (k-NN) y *Regresión Logística* para clasificar sitios web y determinar si son portales legítimos o intentos de Phishing (robo de credenciales). Utilizaremos un dataset público de la UCI con características estructurales de las URLs.

### 1. Preparación del Entorno
Vamos a importar las librerías necesarias. Nota cómo aplicamos nuestra paleta de colores corporativa para asegurar que nuestras gráficas se alineen con la identidad visual.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Configuracion del Branding Académico
COLOR_PRIMARIO = '#1A365D' # Azul Marino
COLOR_ACENTO = '#319795'   # Verde Azulado
COLOR_NEUTRAL = '#4A5568'  # Gris Pizarra

sns.set_palette([COLOR_ACENTO, COLOR_PRIMARIO, COLOR_NEUTRAL])
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Inter', 'Montserrat', 'Arial']

### 2. Carga y Exploración de Datos
Usaremos el Phishing Websites Dataset.

In [ ]:
# Cargamos los datos desde un repositorio en crudo
url_dataset = "https://raw.githubusercontent.com/MarcosAng/Datasets-ML/main/phishing_dataset.csv"
df = pd.read_csv(url_dataset)

# Revisamos las primeras 5 filas
print("--- Muestra del Dataset ---")
df.head()

### 3. La Analogía del Tobogán y de los Vecinos
> **k-NN (Vecinos):** Si encuentras un sitio web y observas a sus 5 sitios web "gemelos" más parecidos (en longitud de URL y SSL), y 4 de ellos resultan ser Phishing, la conclusión de k-NN es que este nuevo sitio también es Phishing.
>
> **Regresión Logística (El Tobogán):** Sumamos puntos de sospecha (ej. no tiene SSL = +10, tiene IP directa = +5). Si la suma es muy alta, te deslizas hacia la derecha del tobogán (100% de probabilidad de ser Phishing). Si la suma es baja, te quedas a la izquierda (Legítimo).

In [ ]:
# Preparamos nuestros datos: X (atributos) e y (objetivo)
# Mapearemos: -1 (Phishing) -> 0 y 1 (Legítimo) -> 1 para mayor compatibilidad con métricas binarias
X = df.drop(columns=['Result'])
y = df['Result'].replace({-1: 0, 1: 1})

# División 80% Entrenamiento, 20% Prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Set de Entrenamiento: {X_train.shape[0]} registros.")
print(f"Set de Prueba: {X_test.shape[0]} registros.")

### 4. Entrenamiento de k-NN

In [ ]:
# Instanciamos k-NN con k=5 vecinos
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print("--- k-NN Report ---")
print(classification_report(y_test, y_pred_knn, target_names=['Phishing (0)', 'Legítimo (1)']))

### 5. Entrenamiento de Regresión Logística

In [ ]:
# Instanciamos Regresión Logística
log_reg = LogisticRegression(C=1.0, max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)

print("--- Logistic Regression Report ---")
print(classification_report(y_test, y_pred_log, target_names=['Phishing (0)', 'Legítimo (1)']))

### 6. Análisis Visual: Matrices de Confusión
En ciberseguridad, la Matriz de Confusión es crucial. Nos dice exactamente cuántos Phishing se escaparon (Falsos Negativos) y cuántos Legítimos fueron bloqueados (Falsos Positivos).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#F7FAFC')

cm_knn = confusion_matrix(y_test, y_pred_knn)
disp_knn = ConfusionMatrixDisplay(cm_knn, display_labels=['Phishing', 'Legítimo'])
disp_knn.plot(ax=ax[0], cmap=sns.light_palette(COLOR_ACENTO, as_cmap=True), colorbar=False)
ax[0].set_title('k-NN (Vecinos)', color=COLOR_PRIMARIO, weight='bold')

cm_log = confusion_matrix(y_test, y_pred_log)
disp_log = ConfusionMatrixDisplay(cm_log, display_labels=['Phishing', 'Legítimo'])
disp_log.plot(ax=ax[1], cmap=sns.light_palette(COLOR_PRIMARIO, as_cmap=True), colorbar=False)
ax[1].set_title('Regresión Logística (Tobogán)', color=COLOR_PRIMARIO, weight='bold')

plt.tight_layout()
plt.show()